# Resume Screening with LLMs

**Goal**: Build a resume scoring system using structured outputs and submit scores to the leaderboard.

## Setup

In [ ]:
import json
from pydantic import BaseModel
from resume_utils import load_resumes, analyze_resume, submit_score, delete_score, delete_team

# --- Configuration ---
OPENROUTER_API_KEY = ""  # Paste your key here
MODEL = "anthropic/claude-sonnet-4-6"
TEAM_NAME = ""  # Set your team name
LEADERBOARD_URL = "http://ai-leaderboard.site"

if not OPENROUTER_API_KEY.strip():
    raise RuntimeError(
        "Please set OPENROUTER_API_KEY above before running this notebook.\n"
        "Get your key from: https://openrouter.ai/keys"
    )

if not TEAM_NAME.strip():
    raise RuntimeError(
        "Please set your TEAM_NAME above before running this notebook."
    )

# Load resume data
resumes = load_resumes("../data/resumes_final.csv")
sample_id = list(resumes.keys())[0]
sample_resume = resumes[sample_id]

print(f"API key configured")
print(f"Model: {MODEL}")
print(f"Team: {TEAM_NAME}")
print(f"Loaded {len(resumes)} resumes")
print(f"Sample resume ID: {sample_id}")
print(f"Resume preview: {sample_resume['Resume_str'][:200]}...")

## Example: Score a Resume and Submit to Leaderboard

In [ ]:
# Define the output structure
class ResumeScore(BaseModel):
    score: int
    reasoning: str

# Score a resume using structured output
prompt = "On a scale of 1 to 100, how good of a fit is this person for a software engineering job?"

result = analyze_resume(
    OPENROUTER_API_KEY,
    prompt,
    sample_resume["Resume_str"],
    ResumeScore,
    model=MODEL,
)

if result["error"]:
    print(f"Error: {result['error']}")
else:
    score = result["result"]["score"]
    reasoning = result["result"]["reasoning"]
    print(f"Resume {sample_id} score: {score}/100")
    print(f"Reasoning: {reasoning}")
    print(f"Tokens used: {result['usage'].get('total_tokens', 0)}")

    # Submit to leaderboard
    response = submit_score(TEAM_NAME, sample_id, score, api_url=LEADERBOARD_URL)
    print(f"\nSubmitted to leaderboard: {response}")

## Clean Up: Remove the Example Submission

In [ ]:
# Remove the single example submission we just made
response = delete_score(TEAM_NAME, sample_id, api_url=LEADERBOARD_URL)
print(f"Deleted: {response}")

# To delete ALL submissions for your team, uncomment:
# response = delete_team(TEAM_NAME, api_url=LEADERBOARD_URL)
# print(f"Deleted all: {response}")